# Brain Tumor Classification – Multiple Models Comparison

Dataset: Brain Tumor MRI (glioma / meningioma / pituitary / notumor)

Models compared:
- Classical ML: SVM, RandomForest, XGBoost (with flattened images or HOG)
- Deep Learning:
  - Simple CNN from scratch
  - EfficientNetB0 (transfer learning)
  - ResNet50 (transfer learning)
  - MobileNetV2 (transfer learning – lightweight)

In [ ]:
!pip install -q tensorflow-addons tensorflow keras==2.15.*  # if needed
!pip install -q xgboost opencv-python-headless scikit-image

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0, ResNet50, MobileNetV2
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import warnings
warnings.filterwarnings('ignore')

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 1. Configuration

In [ ]:
# ──── CHANGE THESE PATHS ────────────────────────────────────────
TRAIN_DIR = "/kaggle/input/brain-tumor-mri-dataset/Training"     # ← most common path on Kaggle
TEST_DIR  = "/kaggle/input/brain-tumor-mri-dataset/Testing"

IMG_SIZE_CLASSICAL = (128, 128)     # for ML models
IMG_SIZE_DL        = (224, 224)     # for transfer learning

BATCH_SIZE = 32
EPOCHS = 25
SEED = 42

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

## 2. Data Generators (for deep learning models)

In [ ]:
datagen_train = ImageDataGenerator(
    preprocessing_function=eff_preprocess,  # we'll change per model
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

datagen_val = ImageDataGenerator(preprocessing_function=eff_preprocess)

## 3. Helper functions

In [ ]:
def build_simple_cnn(input_shape=(224,224,3), num_classes=4):
    model = models.Sequential([
        layers.Conv2D(32, 3, activation='relu', input_shape=input_shape),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def create_transfer_model(base_model_class, preprocess_input, name="model"):
    base = base_model_class(
        include_top=False,
        weights='imagenet',
        input_shape=(224,224,3),
        pooling='avg'
    )
    base.trainable = False

    model = models.Sequential([
        layers.Input(shape=(224,224,3)),
        base,
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(4, activation='softmax')
    ], name=name)

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model, preprocess_input

## 4. Load data for classical ML (flattened or HOG)

In [ ]:
def load_data_for_ml(root_dir, size=(128,128), use_hog=False):
    images = []
    labels = []
    
    for cls in CLASS_NAMES:
        path = os.path.join(root_dir, cls)
        label = CLASS_NAMES.index(cls)
        for img_name in os.listdir(path):
            img_path = os.path.join(path, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            img = cv2.resize(img, size)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            if use_hog:
                from skimage.feature import hog
                fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                         cells_per_block=(2,2), visualize=False)
                images.append(fd)
            else:
                images.append(img.flatten())
                
            labels.append(label)
    
    return np.array(images), np.array(labels)

In [ ]:
print("Loading data for classical ML models...")
X_train_ml, y_train_ml = load_data_for_ml(TRAIN_DIR, IMG_SIZE_CLASSICAL, use_hog=False)
X_test_ml,  y_test_ml  = load_data_for_ml(TEST_DIR,  IMG_SIZE_CLASSICAL, use_hog=False)

print(X_train_ml.shape, y_train_ml.shape)
print(X_test_ml.shape,  y_test_ml.shape)

## 5. Train & Evaluate Classical ML models

In [ ]:
models_ml = {}
results_ml = []

# SVM
svm = SVC(kernel='rbf', probability=True, random_state=SEED)
svm.fit(X_train_ml, y_train_ml)
acc = accuracy_score(y_test_ml, svm.predict(X_test_ml))
f1   = f1_score(y_test_ml, svm.predict(X_test_ml), average='macro')
results_ml.append(('SVM (flat)', acc, f1))
models_ml['SVM'] = svm

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_train_ml, y_train_ml)
acc = accuracy_score(y_test_ml, rf.predict(X_test_ml))
f1  = f1_score(y_test_ml, rf.predict(X_test_ml), average='macro')
results_ml.append(('RandomForest', acc, f1))
models_ml['RF'] = rf

# XGBoost
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=8, random_state=SEED, n_jobs=-1)
xgb_model.fit(X_train_ml, y_train_ml)
acc = accuracy_score(y_test_ml, xgb_model.predict(X_test_ml))
f1  = f1_score(y_test_ml, xgb_model.predict(X_test_ml), average='macro')
results_ml.append(('XGBoost', acc, f1))
models_ml['XGB'] = xgb_model

df_ml = pd.DataFrame(results_ml, columns=['Model','Accuracy','F1-macro'])
print(df_ml.sort_values('F1-macro', ascending=False))

## 6. Deep Learning models

In [ ]:
dl_models = []
histories = {}

# ─── 6.1 Simple CNN ───────────────────────────────────────
cnn = build_simple_cnn()
datagen_train.preprocessing_function = tf.keras.applications.resnet50.preprocess_input
datagen_val.preprocessing_function   = tf.keras.applications.resnet50.preprocess_input

train_gen = datagen_train.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE_DL,
                                              batch_size=BATCH_SIZE, class_mode='categorical', seed=SEED)
test_gen  = datagen_val.flow_from_directory(TEST_DIR, target_size=IMG_SIZE_DL,
                                            batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

chkpt = ModelCheckpoint('best_simple_cnn.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
es    = EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)

hist = cnn.fit(train_gen, epochs=EPOCHS, validation_data=test_gen,
               callbacks=[chkpt, es], verbose=1)

histories['SimpleCNN'] = hist
dl_models.append(('SimpleCNN', cnn, test_gen, 'best_simple_cnn.h5'))

In [ ]:
# ─── 6.2 Transfer Learning models ────────────────────────────────
transfer_configs = [
    (EfficientNetB0, eff_preprocess, "EfficientNetB0"),
    (ResNet50,       resnet_preprocess, "ResNet50"),
    (MobileNetV2,    mob_preprocess,    "MobileNetV2")
]

for base_cls, prep_func, name in transfer_configs:
    print(f"\nTraining {name}...")
    model, prep = create_transfer_model(base_cls, prep_func, name)
    datagen_train.preprocessing_function = prep
    datagen_val.preprocessing_function   = prep

    train_gen = datagen_train.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE_DL,
                                                  batch_size=BATCH_SIZE, class_mode='categorical')
    test_gen  = datagen_val.flow_from_directory(TEST_DIR, target_size=IMG_SIZE_DL,
                                                batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

    chkpt = ModelCheckpoint(f'best_{name}.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
    es    = EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True)

    hist = model.fit(train_gen, epochs=EPOCHS, validation_data=test_gen,
                     callbacks=[chkpt, es])

    histories[name] = hist
    dl_models.append((name, model, test_gen, f'best_{name}.h5'))

## 7. Compare ALL models & save the best one

In [ ]:
final_results = []

# ML models
for name, acc, f1 in results_ml:
    final_results.append({'Model': name, 'Accuracy': acc, 'F1-macro': f1, 'Path': None})

# DL models
for name, model, gen, path in dl_models:
    y_pred = model.predict(gen)
    y_pred_class = np.argmax(y_pred, axis=1)
    y_true = gen.classes

    acc = accuracy_score(y_true, y_pred_class)
    f1  = f1_score(y_true, y_pred_class, average='macro')
    final_results.append({'Model': name, 'Accuracy': acc, 'F1-macro': f1, 'Path': path})

df_final = pd.DataFrame(final_results)
df_final = df_final.sort_values('F1-macro', ascending=False).reset_index(drop=True)
print("\nFinal Ranking:")
print(df_final)

# ─── Save best model ───────────────────────────────────────────
best_row = df_final.iloc[0]
best_model_name = best_row['Model']
best_path = best_row['Path']

if best_path is not None:
    print(f"\nBest model: {best_model_name} → saved as → {best_path}")
    print("You can download it from the output files panel (Kaggle) or right-click in Colab")
else:
    print(f"\nBest model was classical ML: {best_model_name}")
    import joblib
    joblib.dump(models_ml[best_model_name.split()[0]], f"best_model_{best_model_name.split()[0]}.pkl")
    print("Saved classical model as .pkl file")

## 8. Optional – Visualize training history of best DL model

In [ ]:
if best_path is not None:
    name = best_model_name
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(histories[name].history['accuracy'], label='train')
    plt.plot(histories[name].history['val_accuracy'], label='val')
    plt.title(f'{name} – Accuracy')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(histories[name].history['loss'], label='train')
    plt.plot(histories[name].history['val_loss'], label='val')
    plt.title(f'{name} – Loss')
    plt.legend()
    plt.show()